#### Import

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent

SRC_PATH = PROJECT_ROOT/ 'src'
if str(SRC_PATH) not in sys.path: 
    sys.path.append(str(SRC_PATH))

import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from config import (
    PA_TABLE_PARQUET,
    INTERTIES_HOUR_AHEAD_PARQUET,
    INTERTIE_CAPABILITY_PARQUET,
    GENERATION_PARQUET,
    OUTAGES_PARQUET,
    CALENDAR_FEATURES,
    LOAD_WEATHER_FEATURES,
    MARKET_FEATURES,
    RENEWABLE_WEATHER_FEATURES,
    RANDOM_SEED,
)

# Suppress warnings so the outputs in the notebooks stay cleaner. 
warnings.filterwarnings('ignore')
# This handles that annoying output compression where you get '...'
# instead of displaying all columns.
pd.set_option('display.max_columns', None)
# Changes the output width of the jupyter notebook. 
pd.set_option('display.width', 120)
np.random.seed(RANDOM_SEED)

pa_hourly = pd.read_parquet(PA_TABLE_PARQUET)
interties_hour_ahead = pd.read_parquet(INTERTIES_HOUR_AHEAD_PARQUET)
intertie_capability = pd.read_parquet(INTERTIE_CAPABILITY_PARQUET)
generation = pd.read_parquet(GENERATION_PARQUET)
outages = pd.read_parquet(OUTAGES_PARQUET)
calendar = pd.read_parquet(CALENDAR_FEATURES)
weather = pd.read_parquet(LOAD_WEATHER_FEATURES)
market = pd.read_parquet(MARKET_FEATURES)
renewables = pd.read_parquet(RENEWABLE_WEATHER_FEATURES)

#### Dataset Overview

In [2]:
summary = []

datasets = {
    'pa_hourly': pa_hourly,
    'interties': interties_hour_ahead,
    'intertie_capability': intertie_capability,
    'generation': generation,
    'outages': outages,
    'calendar': calendar,
    'weather': weather,
    'market': market,
    'renewables': renewables,
}

for name, df in datasets.items():
    summary.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Start': df['timestamp_utc'].min(),
        'End': df['timestamp_utc'].max(),
        'Duplicate timestamps': df['timestamp_utc'].duplicated().sum(),
        'Missing values': int(df.isna().sum().sum()),
        'Memory (MB)': round(df.memory_usage(deep=True).sum() / 1e6, 2),
    })

summary = pd.DataFrame(summary)

summary

,Dataset,Rows,Columns,Start,End,Duplicate timestamps,Missing values,Memory (MB)
0,pa_hourly,100055,5,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0,0,4.00
1,interties,136583,8,2010-01-01 07:00:00+00:00,2025-08-01 05:00:00+00:00,0,62786,8.74
2,intertie_capability,100033,9,2015-01-01 07:00:00+00:00,2026-05-31 07:00:00+00:00,0,0,7.20
3,generation,100055,61,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0,985430,48.83
4,outages,100033,13,2015-01-01 07:00:00+00:00,2026-05-31 07:00:00+00:00,0,197686,10.40
5,calendar,100776,46,2015-01-01 00:00:00+00:00,2026-06-30 23:00:00+00:00,0,0,19.16
6,weather,87672,73,2015-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00,0,110326,45.68
7,market,100055,264,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0,739383,185.12
8,renewables,100776,51,2015-01-01 00:00:00+00:00,2026-06-30 23:00:00+00:00,0,44295,41.12


In [3]:
# Missing values - column specific.

for name, df in datasets.items():
    missing = df.isna().sum()

    missing = missing[missing > 0]

    print(f'\n{name}')

    if missing.empty: 
        print('No missing values.')
    else: 
        print(missing.sort_values(ascending = False))


pa_hourly
No missing values.

interties
export_mt                    31391
import_mt                    31391
hour_ahead_price_forecast        4
dtype: int64

intertie_capability
No missing values.

generation
gas_fired_steam_system_available     53352
gas_fired_steam_system_generation    53352
gas_fired_steam_maximum_capacity     53352
gas_fired_steam_system_capacity      53352
gas_fired_steam_total_generation     53352
storage_total_generation             51143
storage_system_capacity              51143
storage_system_available             51143
storage_maximum_capacity             51143
storage_system_generation            51143
dual_fuel_system_available           50135
dual_fuel_system_generation          50135
dual_fuel_total_generation           50135
dual_fuel_system_capacity            50135
dual_fuel_maximum_capacity           50135
solar_total_generation               25656
solar_system_available               25656
solar_system_capacity                25656
solar_system_ge

* Hour ahead price forecast only missing 4 values is essentailly complete. 
* Montana may have not been reporting intertie data for parts of the historical record. 
* Generation is a different case, you'd expect missing values as technologies phase-in and phase-out across time.
* Same applies to outages - technologies have to exist to experience outages. 
* Market features dataset inherets alot of missingness from generation. 
* Solar clear sky radiation is missing alot of values during nighttime. 

#### External Data Checks

In [4]:
print(interties_hour_ahead.loc[
    interties_hour_ahead['import_mt'].notna(),
    'timestamp_utc'
].min())

print(interties_hour_ahead.loc[
    interties_hour_ahead['export_mt'].notna(),
    'timestamp_utc'
].min())

2013-08-01 06:00:00+00:00
2013-08-01 06:00:00+00:00


In [5]:
cutoff = pd.Timestamp('2013-08-01 06:00:00+00:00', tz = 'UTC')

mt_missing = (
    interties_hour_ahead.loc[interties_hour_ahead['timestamp_utc'] > cutoff,
        ['import_mt', 'export_mt']].isna().sum()
)

print(mt_missing)

import_mt    0
export_mt    0
dtype: int64


Montana Intertie Data: 
* import_mt and export_mt are missing prior to August 2013. Investigation confirmed these missing values are expected: the Montana-Alberta Tie Line (MATL) entered service in 2013, and AESO public reporting of Montana intertie schedules and transfer capability began around the commissioning period. These missing values therefore represent the absence of an operational/reporting intertie rather than data quality issues. Missing values disappear after 2013-08-01 06:00:00+00:00 UTC. 

In [6]:
generation_coverage = []

for col in generation.columns:
    if col == 'timestamp_utc': 
        continue

    non_missing = generation.loc[generation[col].notna(), 'timestamp_utc']

    generation_coverage.append({
        'Column': col,
        'First non-missing': non_missing.min(),
        'Last non-missing': non_missing.max(),
        'Missing': generation[col].isna().sum(),
    })

generation_coverage = (
    pd.DataFrame(generation_coverage).sort_values('First non-missing')
)

generation_coverage

,Column,First non-missing,Last non-missing,Missing
0,coal_system_generation,2015-01-01 07:00:00+00:00,2024-07-01 05:00:00+00:00,16800
28,other_system_available,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
58,total_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
32,wind_system_available,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
33,coal_system_capacity,2015-01-01 07:00:00+00:00,2024-07-01 05:00:00+00:00,16800
34,cogeneration_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
35,combined_cycle_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
38,hydro_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
39,other_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
40,simple_cycle_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0


Generation Data: 
* Missingness in the generation dataset is highly structured by fuel category. Coal and dual-fuel fields terminate around Alberta’s final coal-to-gas conversions in July 2024, while solar and storage fields begin later as those categories entered AESO reporting. Gas-fired steam and the beginning of dual-fuel coverage likely reflect AESO classification or reporting changes rather than the physical introduction of those technologies. Because all metrics within each category share identical coverage boundaries, the missing values are treated as structurally unavailable rather than randomly missing.


# Notebook 1 - Understanding the Alberta Electricity Market

#### Understanding the Alberta Electricity Market 

Purpose:
This notebook introduces the Alberta electricity market and establishes the operational context for the analyses that follow.
Rather than developing predictive models, the objective is to understand how the system behaves:
* how electricity demand evolves over time
* how supply is provided
* how prices behave
* how generation technologies contribute
* how imports and outages influence operations
* how the market has changed over the study period
The observations developed here form the foundation for later notebooks on market regimes, scarcity, feature engineering, and forecasting.

#### Merge

In [7]:
# =============================================================================
# Merge canonical feature datasets across their common hourly timeframe
# =============================================================================

datasets = {
    "market": market,
    "intertie_capability": intertie_capability,
    "generation": generation,
    "weather": weather,
    "renewables": renewables,
}

merge_key = "timestamp_utc"

duplicate_columns_removed = []

# Start with the first dataframe
first_name, first_df = next(iter(datasets.items()))

master = (
    first_df
    .sort_values(merge_key)
    .copy()
)

# Merge each remaining dataframe
for dataset_name, df in list(datasets.items())[1:]:
    incoming = (
        df
        .sort_values(merge_key)
        .copy()
    )

    # Find shared non-key column names
    overlapping_columns = sorted(
        (set(master.columns) & set(incoming.columns))
        - {merge_key}
    )

    columns_to_drop = []

    for column in overlapping_columns:
        comparison = (
            master[[merge_key, column]]
            .merge(
                incoming[[merge_key, column]],
                on=merge_key,
                how="inner",
                suffixes=("_master", "_incoming"),
                validate="one_to_one",
            )
        )

        master_col = f"{column}_master"
        incoming_col = f"{column}_incoming"

        values_match = (
            comparison[master_col].eq(
                comparison[incoming_col]
            )
            | (
                comparison[master_col].isna()
                & comparison[incoming_col].isna()
            )
        )

        if values_match.all():
            # Keep the version already in master
            columns_to_drop.append(column)

            duplicate_columns_removed.append(
                {
                    "incoming_dataset": dataset_name,
                    "column_removed": column,
                }
            )

        else:
            discrepancies = comparison.loc[
                ~values_match.fillna(False),
                [
                    merge_key,
                    master_col,
                    incoming_col,
                ],
            ]

            print(
                f"\nConflicting duplicate column: {column!r}"
                f"\nIncoming dataset: {dataset_name!r}"
                f"\nMismatched rows: {len(discrepancies):,}"
            )

            display(
                discrepancies.head(50)
            )

            raise ValueError(
                f"Column {column!r} exists in both master and "
                f"{dataset_name!r}, but the values are not identical."
            )

    # Remove identical incoming duplicates before merging
    incoming = incoming.drop(
        columns=columns_to_drop
    )

    master = master.merge(
        incoming,
        on=merge_key,
        how="inner",
        validate="one_to_one",
    )

master = (
    master
    .sort_values(merge_key)
    .reset_index(drop=True)
)

# =============================================================================
# Final audit
# =============================================================================

suffixed_columns = [
    column
    for column in master.columns
    if column.endswith("_x")
    or column.endswith("_y")
]

print(f"Shape: {master.shape}")
print(f"Start: {master[merge_key].min()}")
print(f"End:   {master[merge_key].max()}")
print(
    f"Duplicate timestamps: "
    f"{master[merge_key].duplicated().sum()}"
)
print(
    f"Missing values: "
    f"{master.isna().sum().sum():,}"
)
print(
    f"Identical duplicate columns removed: "
    f"{len(duplicate_columns_removed)}"
)
print(
    f"Unexpected suffixed columns: "
    f"{len(suffixed_columns)}"
)

if duplicate_columns_removed:
    duplicate_audit = pd.DataFrame(
        duplicate_columns_removed
    )

    display(
        duplicate_audit
    )

if suffixed_columns:
    raise ValueError(
        "Unexpected merge suffix columns remain: "
        f"{suffixed_columns}"
    )

Shape: (87665, 452)
Start: 2015-01-01 07:00:00+00:00
End:   2024-12-31 23:00:00+00:00
Duplicate timestamps: 0
Missing values: 1,236,776
Identical duplicate columns removed: 2
Unexpected suffixed columns: 0


,incoming_dataset,column_removed
0,weather,hour_alberta
1,weather,month_alberta


In [8]:
missing_audit = (
    master
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

missing_audit["missing_pct"] = (
    missing_audit["missing_rows"]
    / len(master)
    * 100
)

missing_audit = (
    missing_audit
    .query("missing_rows > 0")
    .sort_values(
        ["missing_pct", "missing_rows"],
        ascending=False,
    )
)

print(
    f"Columns with missing values: "
    f"{len(missing_audit):,} / {master.shape[1]:,}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(missing_audit)


Columns with missing values: 202 / 452


,missing_rows,missing_pct
load_weighted_heat_index_c,86817,99.032681
gas_fired_steam_outage,53352,60.858952
gas_fired_steam_system_generation,53352,60.858952
gas_fired_steam_total_generation,53352,60.858952
gas_fired_steam_system_available,53352,60.858952
gas_fired_steam_system_capacity,53352,60.858952
gas_fired_steam_maximum_capacity,53352,60.858952
storage_outage,51143,58.339132
storage_outage_mw,51143,58.339132
storage_system_generation,51143,58.339132


* Historical source coverage: Gas-fired steam, storage, dual fuel, solar, and coal variables have substantial missing blocks because those technologies were not consistently reported across the full study period or their datasets begin later than the master dataset.
* Conditionally defined weather variables:
    - load_weighted_heat_index_c is approximately 99% missing because heat index is only calculated for regional conditions with temperature of at least 26.7°C and relative humidity of at least 40%.
    - load_weighted_wind_chill_c is approximately 27% missing because wind chill is only calculated when temperature is at or below 10°C and wind speed is at least 4.8 km/h.
    - These values are intentionally undefined outside their applicable meteorological regimes and should not be treated as ordinary missing observations.
* Solar clear-sky ratio: Missing values are concentrated during nighttime and low-radiation hours, when clear-sky radiation is zero and the ratio cannot be calculated.
* Lagged and change features: Missing values at the beginning of the dataset are expected mechanical consequences of feature construction. For example, a 24-hour lag produces 24 missing rows and a 168-hour lag produces 168 missing rows.
* Small isolated gaps: A few variables contain only one or several missing observations. These are negligible relative to the full dataset and can be handled during construction of the model-ready sample.

#### Section 1 - How Has Alberta Changed?

Alberta’s electricity market changed materially during the study period. Demand evolved, renewable capacity expanded, coal generation declined, gas-fired generation assumed a larger role, and new technologies entered the supply mix.  
This section establishes how the market’s scale and structure changed before examining shorter-term operating behaviour. Annual results are interpreted cautiously where a year contains only partial data.

##### 1.1 - How Has Electricity Demand Changed?

In [9]:
# =============================================================================
# Prepare the demand-analysis sample
# =============================================================================

load_col = "ail_mw"

required_columns = [
    "timestamp_utc",
    "year_alberta",
    "month_alberta",
    load_col,
]

missing_columns = [
    column
    for column in required_columns
    if column not in market.columns
]

if missing_columns:
    raise KeyError(
        "The following calendar or load columns are missing from market: "
        f"{missing_columns}"
    )

demand = (
    market.loc[:, required_columns]
    .dropna(subset=[load_col])
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
    .copy()
)

print(f"Rows:  {len(demand):,}")
print(f"Start: {demand['timestamp_utc'].min()}")
print(f"End:   {demand['timestamp_utc'].max()}")

Rows:  100,055
Start: 2015-01-01 07:00:00+00:00
End:   2026-06-01 05:00:00+00:00


In [10]:
# =============================================================================
# Audit annual coverage using Alberta local calendar years
# =============================================================================

annual_demand_coverage = (
    demand
    .groupby("year_alberta")
    .agg(
        first_timestamp=("timestamp_utc", "min"),
        last_timestamp=("timestamp_utc", "max"),
        observed_hours=("timestamp_utc", "size"),
        unique_hours=("timestamp_utc", "nunique"),
    )
)

annual_demand_coverage["expected_hours"] = (
    annual_demand_coverage.index.map(
        lambda year: (
            pd.Timestamp(f"{year}-12-31").dayofyear * 24
        )
    )
)

annual_demand_coverage["missing_hours"] = (
    annual_demand_coverage["expected_hours"]
    - annual_demand_coverage["unique_hours"]
)

annual_demand_coverage["coverage_pct"] = (
    annual_demand_coverage["unique_hours"]
    / annual_demand_coverage["expected_hours"]
    * 100
)

annual_demand_coverage["fully_complete"] = (
    annual_demand_coverage["unique_hours"]
    == annual_demand_coverage["expected_hours"]
)

annual_demand_coverage["substantially_complete"] = (
    annual_demand_coverage["coverage_pct"] >= 95
)

annual_demand_coverage

,first_timestamp,last_timestamp,observed_hours,unique_hours,expected_hours,missing_hours,coverage_pct,fully_complete,substantially_complete
year_alberta,,,,,,,,,
2015,2015-01-01 07:00:00+00:00,2016-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True
2016,2016-01-01 07:00:00+00:00,2017-01-01 06:00:00+00:00,8784,8784,8784,0,100.000000,True,True
2017,2017-01-01 07:00:00+00:00,2018-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True
2018,2018-01-01 07:00:00+00:00,2019-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True
2019,2019-01-01 07:00:00+00:00,2020-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True
2020,2020-01-01 07:00:00+00:00,2021-01-01 06:00:00+00:00,8784,8784,8784,0,100.000000,True,True
2021,2021-01-01 07:00:00+00:00,2022-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True
2022,2022-01-01 07:00:00+00:00,2023-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True
2023,2023-01-01 07:00:00+00:00,2024-01-01 06:00:00+00:00,8760,8760,8760,0,100.000000,True,True


In [11]:
complete_demand_years = annual_demand_coverage.index[
    annual_demand_coverage["fully_complete"]
].tolist()

substantially_complete_demand_years = annual_demand_coverage.index[
    annual_demand_coverage["substantially_complete"]
].tolist()

print("Fully complete Alberta years:", complete_demand_years)
print(
    "Substantially complete Alberta years:",
    substantially_complete_demand_years,
)

Fully complete Alberta years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Substantially complete Alberta years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [12]:
# =============================================================================
# Annual Alberta Internal Load summary
# =============================================================================

annual_load = (
    demand.groupby('year_alberta')[load_col].agg(
        average_load_mw='mean',
        median_load_mw = 'median',
        peak_load_mw = 'max',
        minimum_load_mw = 'min',
        observed_hours = 'size',
    )
)

# Since the data are hourly, summing MW observations produces MWh. 
# Dividing by 1,000,000 converts MWh to TWh.
annual_load['annual_energy_twh'] = (
    demand.groupby('year_alberta')[load_col].sum() / 1_000_000
)

annual_load['average_load_growth_pct'] = (
    annual_load['average_load_mw'].pct_change() * 100
)

annual_load = annual_load.join(
    annual_demand_coverage[
        [
            'coverage_pct',
            'fully_complete',
            'substantially_complete',
        ]
    ]
)

annual_load.round(2)

,average_load_mw,median_load_mw,peak_load_mw,minimum_load_mw,observed_hours,annual_energy_twh,average_load_growth_pct,coverage_pct,fully_complete,substantially_complete
year_alberta,,,,,,,,,,
2015,9161.75,9168.0,11229,7203,8760,80.26,NaN,100.00,True,True
2016,9057.37,9080.0,11458,6595,8784,79.56,-1.14,100.00,True,True
2017,9426.08,9401.0,11473,7600,8760,82.57,4.07,100.00,True,True
2018,9740.86,9749.5,11697,7819,8760,85.33,3.34,100.00,True,True
2019,9694.67,9677.0,11471,8024,8760,84.93,-0.47,100.00,True,True
2020,9462.05,9437.0,11698,7579,8784,83.11,-2.40,100.00,True,True
2021,9727.59,9653.5,11729,7976,8760,85.21,2.81,100.00,True,True
2022,9882.61,9851.0,12193,8110,8760,86.57,1.59,100.00,True,True
2023,9850.81,9847.0,11572,7873,8760,86.29,-0.32,100.00,True,True


Alberta electricity demand increased over the 2015–2025 period. Average load rose from approximately 9.16 GW in 2015 to 10.32 GW in 2025, equivalent to total growth of roughly 12.6% and compound annual growth of approximately 1.2%. Annual energy demand increased from 80.3 TWh to 90.4 TWh over the same period.

Growth was not continuous. Average load declined in 2016, 2019, 2020, and 2023, indicating that the long-run trend contains meaningful annual variation. Peak demand increased slightly more than average demand over the full period, while minimum load also moved materially higher. This suggests that demand growth may reflect a broad upward shift in system load rather than growth confined only to extreme hours.

Results for 2026 are excluded from direct annual comparisons because the available observations cover only January through May. The elevated partial-year average should not be interpreted as full-year growth because the observed months have a different seasonal composition from a complete calendar year.

##### 1.2 - How Has Generation Mix Changed?

In [13]:
# =============================================================================
# Generation technology mapping
# =============================================================================

GENERATION_COLUMNS = {
    "Coal": "coal_system_generation",
    "Cogeneration": "cogeneration_system_generation",
    "Combined cycle": "combined_cycle_system_generation",
    "Dual fuel": "dual_fuel_system_generation",
    "Gas-fired steam": "gas_fired_steam_system_generation",
    "Hydro": "hydro_system_generation",
    "Other": "other_system_generation",
    "Simple cycle": "simple_cycle_system_generation",
    "Solar": "solar_system_generation",
    "Storage": "storage_system_generation",
    "Wind": "wind_system_generation",
}

invalid_generation_columns = {
    technology: column
    for technology, column in GENERATION_COLUMNS.items()
    if column not in master.columns
}

if invalid_generation_columns:
    raise KeyError(
        "Invalid generation-column mappings: "
        f"{invalid_generation_columns}"
    )

print("All generation columns were found.")

All generation columns were found.


In [14]:
print("market:")
print("year_alberta" in market.columns)
print("month_alberta" in market.columns)
print("coal_system_generation" in market.columns)

print("\nmaster:")
print("year_alberta" in master.columns)
print("month_alberta" in master.columns)
print("coal_system_generation" in master.columns)

market:
True
True
False

master:
True
True
True
